In [ ]:
import os
from tqdm import tqdm
import torch
from torchvision import transforms
from diffusers import AutoencoderKL
from torch.utils.data import DataLoader, Dataset

# --- 설정 ---
device = 'cuda'
latent_dir = './latents/valid'
os.makedirs(latent_dir, exist_ok=True)

batch_size = 1024  # 원하는 배치 크기로 조절

# 이미지 전처리
transform = transforms.Compose([
    transforms.ToTensor(),
    # 필요하면 Normalize 등 추가
])

# HuggingFace 원본 데이터셋 래퍼
from datasets import load_dataset
raw_hf = load_dataset("benjamin-paine/imagenet-1k-64x64", split="validation")

class HFBatchDataset(Dataset):
    def __init__(self, hf_ds, transform):
        self.ds = hf_ds
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        img = self.transform(ex["image"])
        label = ex["label"]
        return img, label

# DataLoader
batch_ds = HFBatchDataset(raw_hf, transform)
loader   = DataLoader(batch_ds,
                      batch_size=batch_size,
                      shuffle=False,
                      num_workers=8,
                      pin_memory=True)

# VAE 모델 로드
vae = AutoencoderKL.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="vae"
).to(device)
vae.eval()

# 전체 인덱스 카운터
global_idx = 0

# 배치 단위로 latent 저장
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Precomputing latents"):
        imgs = imgs.to(device)                            # [B, C, H, W]
        out  = vae.encode(imgs)
        zs   = out["latent_dist"].sample() * 0.18215       # [B, latent_dim, h, w]
        zs   = zs.cpu()

        # 배치 내 각 샘플별로 파일로 저장
        for i in range(zs.shape[0]):
            torch.save({
                "z": zs[i],
                "label": int(labels[i])
            }, os.path.join(latent_dir, f"{global_idx:07d}.pt"))
            global_idx += 1

print(f"Finished! Saved {global_idx} latent files to {latent_dir}")


In [ ]:
import os
from tqdm import tqdm
import torch
from torchvision import transforms
from diffusers import AutoencoderKL
from torch.utils.data import DataLoader, Dataset

# --- 설정 ---
device = 'cuda'
latent_dir = './latents/train'
os.makedirs(latent_dir, exist_ok=True)

batch_size = 1024  # 원하는 배치 크기로 조절

# 이미지 전처리
transform = transforms.Compose([
    transforms.ToTensor(),
    # 필요하면 Normalize 등 추가
])

# HuggingFace 원본 데이터셋 래퍼
from datasets import load_dataset
raw_hf = load_dataset("benjamin-paine/imagenet-1k-64x64", split="train")

class HFBatchDataset(Dataset):
    def __init__(self, hf_ds, transform):
        self.ds = hf_ds
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        img = self.transform(ex["image"])
        label = ex["label"]
        return img, label

# DataLoader
batch_ds = HFBatchDataset(raw_hf, transform)
loader   = DataLoader(batch_ds,
                      batch_size=batch_size,
                      shuffle=False,
                      num_workers=8,
                      pin_memory=True)

# VAE 모델 로드
vae = AutoencoderKL.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="vae"
).to(device)
vae.eval()
vae = torch.compile(vae)

# 전체 인덱스 카운터
global_idx = 0

# 배치 단위로 latent 저장
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Precomputing latents"):
        imgs = imgs.to(device)                            # [B, C, H, W]
        out  = vae.encode(imgs)
        zs   = out["latent_dist"].sample() * 0.18215       # [B, latent_dim, h, w]
        zs   = zs.cpu()

        # 배치 내 각 샘플별로 파일로 저장
        for i in range(zs.shape[0]):
            torch.save({
                "z": zs[i],
                "label": int(labels[i])
            }, os.path.join(latent_dir, f"{global_idx:07d}.pt"))
            global_idx += 1

print(f"Finished! Saved {global_idx} latent files to {latent_dir}")

In [ ]:
import numpy as np
import torch
import os

# 예시 텐서
z = torch.randn(1024, 4, 8, 8).cpu()  # float16
label = torch.randint(0, 1000, (1024,))

# .pt 저장
torch.save({"z": z, "label": label}, "sample.pt")
pt_size = os.path.getsize("sample.pt") / 1024 / 1024

# .npz 저장
np.savez_compressed("sample.npz", z=z.numpy(), label=label.numpy())
npz_size = os.path.getsize("sample.npz") / 1024 / 1024

print(f".pt size:  {pt_size:.2f} MB")
print(f".npz size: {npz_size:.2f} MB")

In [ ]:
# Transform 정의
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize([0.5], [0.5])
])

# HuggingFace Dataset → PyTorch Dataset으로 감싸기
class HFDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = example["image"]
        label = example["label"]
        if self.transform:
            image = self.transform(image)
        return {
            'image': image,
            'label': label
        }

# 데이터셋 로딩
dataset = load_dataset("benjamin-paine/imagenet-1k-64x64")

# train/val wrapping
train_dataset = HFDatasetWrapper(dataset["train"], transform=transform)
val_dataset = HFDatasetWrapper(dataset["validation"], transform=transform)

# DataLoader 생성
train_dataloader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=8, pin_memory=True)
valid_dataloader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=8, pin_memory=True)

print("train dataloader len : ", len(train_dataloader))
print("valid dataloader len : ", len(valid_dataloader))
# show_tensor_image(next(iter(valid_dataloader))['image'][0])